In [1]:
from dataclasses import dataclass
import os

@dataclass(frozen=True)
class Provider:
    """One Provider to reliabu=ily route requests across all inferennce providers"""
    name:str
    env_var:str
    is_free:bool
    base_url:str
    model:str

PROVIDERS = [
    Provider("OpenAI","OPENAI_API_KEY",True,None,"gpt-4o-mini"),
    Provider("Groq","GROQ_API_KEY",True,"https://api.groq.com/openai/v1","gpt-oss-12ob"),
]

def selector_provider()-> Provider:
    for provider in PROVIDERS:
        if(os.getenv(provider.env_var)):
            return provider
    expected = ", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(f"No Provider key Set,add one of {expected} to your environemnt variable")


def build_client(provider:Provider):
    from openai import OpenAI

    api_key = os.getenv(provider.env_var)
    if provider.base_url is None:
        return OpenAI(api_key=api_key)

    return OpenAI(api_key=api_key,base_url=provider.base_url)

def have_any_keys()-> bool:
    return any(os.getenv(p.env_var)  for p in PROVIDERS)

print("Found a provider key."if have_any_keys() else "No Provide key found")


Found a provider key.


In [2]:
def llm_reply(prompt:str)-> str:
    provider  = selector_provider()
    print(f"Using {provider.name} provider")
    client = build_client(provider)
    result  = client.chat.completions.create(
        model=provider.model,
        max_tokens=200,
        messages=[{
            'role':"user",
            "content":prompt
        }]
    )
    return result.choices[0].message.content

In [3]:
prompt = "Who was the PM of India 2010"

try:
    print(llm_reply(prompt))
except Exception as e:
    print(f"Error:{e}")

Using OpenAI provider
In 2010, the Prime Minister of India was Manmohan Singh. He served as Prime Minister from 2004 to 2014, representing the Indian National Congress party.
